# 06 - IEEE-CIS Logical Rule Ablation

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ["THESIS_QUICK_RUN"] = "0"
    os.environ["THESIS_SYNTHETIC_FALLBACK"] = "0"
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

Cloning into '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection'...


{'project_root': '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection', 'git_commit': '0f8234cb38cff4871ead148944284527e43be5c9', 'quick_run': False, 'synthetic_fallback': False, 'kaggle': True}


## Thiết lập

Ablation giữ nguyên frozen IEEE predictor, calibrated probabilities và threshold.
Chỉ rule subset thay đổi giữa các điều kiện.

In [2]:
EXPECTED_DATASET = "ieee_cis"
# EXPECTED_DATASET = "baf"

manifest_paths = list(
    Path("/kaggle/input").glob("**/frozen_reference_manifest.json")
)

matching_manifests = []

for manifest_path in manifest_paths:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    if manifest.get("dataset_name") == EXPECTED_DATASET:
        artifact_path = manifest_path.parent / manifest["artifact_file"]

        print({
            "dataset": manifest.get("dataset_name"),
            "model": manifest.get("model"),
            "reference_seed": manifest.get("reference_seed"),
            "quick_run": manifest.get("quick_run"),
            "manifest": str(manifest_path),
            "artifact": str(artifact_path),
            "artifact_exists": artifact_path.exists(),
        })

        if artifact_path.exists():
            matching_manifests.append(manifest_path)

assert len(matching_manifests) == 1, (
    f"Expected exactly one valid artifact for {EXPECTED_DATASET}, "
    f"found {len(matching_manifests)}. Check Kaggle Add Input."
)

print(f"Artifact preflight passed for {EXPECTED_DATASET}.")

{'dataset': 'ieee_cis', 'model': 'xgboost', 'reference_seed': 42, 'quick_run': False, 'manifest': '/kaggle/input/notebooks/giahuytranviet/02-ieee-cis-model-benchmarks-ipynb/thesis_outputs/02_ieee_cis_model_benchmarks/frozen_reference_manifest.json', 'artifact': '/kaggle/input/notebooks/giahuytranviet/02-ieee-cis-model-benchmarks-ipynb/thesis_outputs/02_ieee_cis_model_benchmarks/frozen_reference_artifact.npz', 'artifact_exists': True}
Artifact preflight passed for ieee_cis.


In [3]:
from src.artifacts import assert_frozen_alignment, load_frozen_reference_artifact
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data

config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
artifact = load_frozen_reference_artifact(
    "ieee_cis", expected_config=config,
    search_roots=[OUTPUT_BASE / "02_ieee_cis_model_benchmarks", *INPUT_ROOTS],
)
assert_frozen_alignment(artifact, prepared.y_validation, prepared.y_test)
if bool(artifact["manifest"]["quick_run"]) != QUICK_RUN:
    raise ValueError("Notebook mode and frozen artifact quick_run flag do not match")
probabilities = artifact["test_probability"]
threshold = float(artifact["manifest"]["threshold"])
print({
    "data_source": data_source,
    "reference_model": artifact["manifest"]["model"],
    "reference_seed": artifact["manifest"]["reference_seed"],
    "calibration_method": artifact["manifest"]["calibration_method"],
    "threshold": threshold,
    "artifact": str(artifact["artifact_path"]),
})

{'data_source': 'real', 'reference_model': 'xgboost', 'reference_seed': 42, 'calibration_method': 'isotonic', 'threshold': 0.08823529411764706, 'artifact': '/kaggle/input/notebooks/giahuytranviet/02-ieee-cis-model-benchmarks-ipynb/thesis_outputs/02_ieee_cis_model_benchmarks/frozen_reference_artifact.npz'}


In [4]:
from src.logic import FraudRuleEngine

output_dir = OUTPUT_BASE / "06_ieee_cis_rule_ablation"
output_dir.mkdir(parents=True, exist_ok=True)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
truth = engine.evaluate(prepared.test_frame)
predicted_alert = probabilities >= threshold
activation = float(config["logic"]["activation_threshold"])

def score_subset(name, columns, activation_threshold=activation):
    rule_score = truth[columns].max(axis=1).to_numpy(float) if columns else np.zeros(len(truth))
    explained = rule_score >= activation_threshold
    explained_alert = explained & predicted_alert
    explained_precision = prepared.y_test[explained_alert].mean() if explained_alert.any() else 0.0
    base_precision = prepared.y_test[predicted_alert].mean() if predicted_alert.any() else 0.0
    return {
        "ablation": name, "activation_threshold": activation_threshold, "rule_count": len(columns),
        "coverage_all": explained.mean(),
        "coverage_alerts": explained_alert.sum() / max(predicted_alert.sum(), 1),
        "explained_alert_precision": explained_precision,
        "precision_gain": explained_precision - base_precision,
        "prediction_rule_consistency": (predicted_alert == explained).mean(),
    }

## Results

In [5]:
all_rules = truth.columns.tolist()
rows = [score_subset("full_rule_set", all_rules), score_subset("no_rules", [])]
rows += [score_subset(f"only:{rule}", [rule]) for rule in all_rules]
rows += [score_subset(f"without:{rule}", [item for item in all_rules if item != rule]) for rule in all_rules]
rows += [score_subset("full_rule_set_sensitivity", all_rules, value) for value in (0.50, 0.60, 0.70, 0.80)]
ablation = pd.DataFrame(rows)
display(ablation.round(4))
ablation.to_csv(output_dir / "ieee_rule_ablation.csv", index=False)

,ablation,activation_threshold,rule_count,coverage_all,coverage_alerts,explained_alert_precision,precision_gain,prediction_rule_consistency
0,full_rule_set,0.6,5,0.2295,0.2848,0.3179,0.0593,0.7375
1,no_rules,0.6,0,0.0000,0.0000,0.0000,-0.2585,0.9232
2,only:high_transaction_amount,0.6,1,0.0435,0.0802,0.2161,-0.0424,0.8920
3,only:unusual_transaction_hour,0.6,1,0.1513,0.1734,0.2458,-0.0128,0.7985
4,only:high_amount_and_device_risk,0.6,1,0.0008,0.0062,0.8810,0.6224,0.9233
5,only:high_amount_and_email_risk,0.6,1,0.0000,0.0000,0.0000,-0.2585,0.9232
6,only:high_velocity_proxy,0.6,1,0.0465,0.0450,0.7974,0.5389,0.8836
7,without:high_transaction_amount,0.6,4,0.1909,0.2168,0.3573,0.0988,0.7656
8,without:unusual_transaction_hour,0.6,4,0.0899,0.1285,0.4371,0.1785,0.8530
9,without:high_amount_and_device_risk,0.6,4,0.2291,0.2816,0.3116,0.0531,0.7373


## Takeaways

In [6]:
full = ablation.query("ablation == 'full_rule_set'").iloc[0]
best = ablation[ablation["ablation"].str.startswith("without:")].sort_values("precision_gain", ascending=False).iloc[0]
display(Markdown(
    f"- Full-set alert coverage: **{full['coverage_alerts']:.3f}**.\n"
    f"- Full-set precision gain: **{full['precision_gain']:.3f}**.\n"
    f"- Best leave-one-out condition: **{best['ablation']}**.\n"
    "- Any change is attributable to the rule subset because predictor outputs are frozen."
))

- Full-set alert coverage: **0.285**.
- Full-set precision gain: **0.059**.
- Best leave-one-out condition: **without:unusual_transaction_hour**.
- Any change is attributable to the rule subset because predictor outputs are frozen.